### Cell 1: Imports & Environment Setup
Install required libraries if you haven't already

In [1]:
!pip install faiss-cpu sentence-transformers transformers PyPDF2

In [2]:
# Local directories to hold models and uploads
import os
from pathlib import Path

HOME = Path.home()

MODELS_DIR = HOME / "ai_models"
UPLOADS_DIR = HOME / "ai_uploads"
LOGS_DIR = "../logs"

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(UPLOADS_DIR, exist_ok=True)

In [3]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sentence_transformers import SentenceTransformer
from collections import Counter

# --- 1. LOAD AND PREPROCESS WINDOWS CSV ---
windows_log_path = "../logs/Windows_Security/_PC1.csv"  # Adjust path as needed

print(f"Checking for Windows log file at: {os.path.abspath(windows_log_path)}")
if os.path.exists(windows_log_path):
    # Read the Windows Security CSV
    # Using python engine or handling multi-line quoted fields gracefully
    df_win = pd.read_csv(windows_log_path, on_bad_lines='skip')
    df_win.columns = [col.strip() for col in df_win.columns]
    print(f"Successfully loaded {len(df_win)} Windows event rows.")
else:
    print("⚠️ File not found. Please check your path.")
    # Create a dummy DataFrame for demonstration if file isn't downloaded yet
    df_win = pd.DataFrame({
        'DateTime': ['24.11.2024 16:49:48', '24.11.2024 16:50:00'],
        'Source': ['Microsoft-Windows-Security-Auditing', 'Microsoft-Windows-Security-Auditing'],
        'EventID': [4798, 4625],
        'Category': ['User Account Management', 'Logon'],
        'Message': ['A users local group membership was enumerated.', 'An account failed to log on. Failure Reason: Bad password.']
    })


Checking for Windows log file at: /home/jovyan/AI_examples/logs/Windows_Security/_PC1.csv
Successfully loaded 27304 Windows event rows.


In [4]:
# Check for GPU (cuda) or CPU
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024**3),
            1
        ),
        "GB"
    )

CUDA: False


### Standardize data

In [5]:
# Limit training lines by eventID
# Minimizes bias and speeds up training

# --- STEP 2: Cap by Event ID BEFORE Categorization ---
MAX_PER_EVENT_ID = 50  # Never let a single Event ID contribute excessive rows

trimmed_chunks = []
for event_id, group in df_win.groupby('EventID'):
    # Sample up to MAX_PER_EVENT_ID rows for this specific Event ID (or take all if fewer)
    if len(group) <= MAX_PER_EVENT_ID:
        trimmed_chunks.append(group)
    else:
        sample_size = min(len(group), MAX_PER_EVENT_ID)
        sampled_group = group.sample(n=sample_size, random_state=42)
        trimmed_chunks.append(sampled_group)

# Recombine into a fresh DataFrame with a clean, standard index
df_win_trimmed = pd.concat(trimmed_chunks, ignore_index=True)

print(f"Dataset reduced to {len(df_win_trimmed)} rows via Event ID capping.")

# --- STEP 3: Now apply your normalization and categorization ---
df_win_trimmed['normalized_log'] = (
    "Source: " + df_win_trimmed['Source'].astype(str) + " | " +
    "EventID: " + df_win_trimmed['EventID'].astype(str) + " | " +
    "Category: " + df_win_trimmed['Category'].astype(str) + " | " +
    "Event: " + df_win_trimmed['Event'].astype(str).str[:300]
)

df_win = df_win_trimmed

Dataset reduced to 731 rows via Event ID capping.


In [6]:
# Clean column names
df_win.columns = [col.strip() for col in df_win.columns]

# Create a unified text field for embedding (combining Event ID, Category, and Message)
df_win['normalized_log'] = (
    "EventID: " + df_win['EventID'].astype(str) + " | " +
    "Category: " + df_win['Category'].astype(str) + " | " +
    df_win.get('Message', df_win.iloc[:, -1]).astype(str)
)

### --- 2. ASSIGN CLASSROOM TAXONOMY ---

In [7]:
def assign_windows_category(row):
    event_id = str(row['EventID'])
    category_text = str(row['Category']).lower()
    msg_text = str(row.get('Event', '')).lower()
    
    # 1. Security (Failed logons, lockouts, threat indicators)
    if event_id in ["4625", "4740", "4719"] or "fail" in msg_text or "audit failure" in msg_text:
        return "Security"
    
    # 2. Authentication (Successful logons, logoffs, sessions)
    elif event_id in ["4624", "4634", "4648"] or "logon" in category_text:
        return "Authentication"
    
    # 3. Configuration (Group enumeration, object auditing changes like 4907/4799, credentials)
    elif event_id in ["4798", "4799", "5379", "4720", "4726", "4907", "4688"] or "account management" in category_text or "audit" in category_text:
        return "Configuration"
    
    # 4. Service activity
    elif event_id in ["4697"] or "service" in category_text:
        return "Service"
        
    # 5. Network categories (Primarily populated by router/switch logs, but safe to include)
    elif any(term in msg_text or term in category_text for term in ["routing", "bgp", "ospf", "route"]):
        return "Routing"
    elif any(term in msg_text or term in category_text for term in ["switch", "vlan", "trunk"]):
        return "Switching"
    elif "interface" in msg_text or "interface" in category_text:
        return "Interface"
        
    # 6. Plain Text (True fallback for unclassified logs)
    else:
        return "Plain Text"

In [8]:
df_win['category'] = df_win.apply(assign_windows_category, axis=1)

print("\nWindows Category Distribution:")
print(df_win['category'].value_counts())


Windows Category Distribution:
category
Configuration     337
Authentication    200
Plain Text        189
Service             3
Security            2
Name: count, dtype: int64


### --- 3. GENERATE EMBEDDINGS & TRAIN CATEGORIZER ---

In [9]:
# --- 3. GENERATE EMBEDDINGS & TRAIN CATEGORIZER ---
print("\nLoading embedding model...")
model = SentenceTransformer(
    'all-MiniLM-L6-v2',
    cache_folder = MODELS_DIR,
    device = DEVICE
)

print("Generating vector embeddings for Windows logs...")
X_win = model.encode(df_win['normalized_log'].tolist(), show_progress_bar=True)
y_win = df_win['category'].to_numpy()

# Train/Test Split & Classifier
if len(df_win) > 5:
    X_w_train, X_w_test, y_w_train, y_w_test = train_test_split(
        X_win, y_win, test_size=0.2, random_state=42, stratify=y_win if len(np.unique(y_win)) > 1 else None
    )
    
    win_classifier = LogisticRegression(max_iter=1000)
    win_classifier.fit(X_w_train, y_w_train)
    
    y_w_pred = win_classifier.predict(X_w_test)
    print("\n--- Windows Category Model Evaluation ---")
    print(classification_report(y_w_test, y_w_pred, zero_division=0))
else:
    win_classifier = LogisticRegression(max_iter=1000)
    win_classifier.fit(X_win, y_win)


Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating vector embeddings for Windows logs...


Batches:   0%|          | 0/23 [00:00<?, ?it/s]


--- Windows Category Model Evaluation ---
                precision    recall  f1-score   support

Authentication       1.00      1.00      1.00        40
 Configuration       0.99      1.00      0.99        68
    Plain Text       0.97      0.97      0.97        38
       Service       0.00      0.00      0.00         1

      accuracy                           0.99       147
     macro avg       0.74      0.74      0.74       147
  weighted avg       0.98      0.99      0.98       147



### --- 4. BINARY THREAT DETECTION (Benign vs Malicious) ---

In [10]:
# Create synthetic malicous logs

import random

# Building blocks for synthetic threat logs
malicious_event_ids = [4688, 4625, 4720, 4697, 4719]
tools = ["mimikatz.exe", "psexec.exe", "procdump.exe", "sharphound.exe", "cobalt_strike.exe", "rubeus.exe"]
parent_processes = ["cmd.exe", "powershell.exe", "svchost.exe", "explorer.exe"]
suspicious_ips = ["192.168.1.150", "10.0.0.99", "172.16.4.20", "203.0.113.5", "45.33.32.156"]
fake_accounts = ["BackdoorAdmin", "ShadowUser", "TempSvc", "AuditBypass", "Guest_Ext"]
services = ["PersistenceAgent", "WinUpdateHook", "RemoteExecSvc", "SystemBackdoor"]

synthetic_malicious_samples = []

# Generate 50 unique synthetic malicious variations
for _ in range(50):
    eid = random.choice(malicious_event_ids)
    tool = random.choice(tools)
    parent = random.choice(parent_processes)
    ip = random.choice(suspicious_ips)
    account = random.choice(fake_accounts)
    service = random.choice(services)
    
    # Format templates matching your exact normalization schema
    if eid == 4688:
        event = f"A new process has been created: C:\\Temp\\{tool} spawned by {parent} with command line: -nop -w hidden"
        cat = "Detailed Tracking"
    elif eid == 4625:
        event = f"An account failed to log on for user {account} from remote source IP {ip}. Reason: Bad password or unknown user."
        cat = "Security"
    elif eid == 4720:
        event = f"A user account was created: Target Account Name: {account} created by suspicious process {parent}."
        cat = "Account Management"
    elif eid == 4697:
        event = f"A service was installed in the system: Service Name: {service} | File Path: C:\\Windows\\Temp\\{tool}"
        cat = "Service Installation"
    else:
        event = f"System audit policy was changed. Security settings and logging capabilities were disabled by account {account}."
        cat = "Policy Change"
        
    log_string = f"Source: Microsoft-Windows-Security-Auditing, EventID: {eid}, Category: {cat}, Event: {event}"
    synthetic_malicious_samples.append(log_string)

# Combine with your original benign samples
# benign_samples = df_win['normalized_log'].tolist()
# all_win_logs = benign_samples + synthetic_malicious_samples
# win_labels = [0] * len(benign_samples) + [1] * len(synthetic_malicious_samples)

print(f"Generated {len(synthetic_malicious_samples)} synthetic malicious samples.")
# print(f"Total training dataset size: {len(all_win_logs)} logs.")

Generated 50 synthetic malicious samples.


In [11]:

# Simulate or label known malicious Windows Event patterns (e.g., Failed Logon 4625, Credential Dumping)
benign_samples = df_win['normalized_log'].tolist()

# Standardize malicious samples to match the exact same string format as benign logs
malicious_windows_samples = [
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4625, Category: Security, Event: An account failed to log on. Sub Status: 0xC000006A (Bad Password)",
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4740, Category: Account Lockout, Event: A user account was locked out due to repeated bad passwords.",
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4688, Category: Detailed Tracking, Event: A new process has been created: C:\\Windows\\System32\\cmd.exe spawned by mimikatz.exe",
    "Source: Microsoft-Windows-Security-Auditing, EventID: 1102, Category: Log Management, Event: The audit log was cleared by an unauthorized user."
]

all_win_logs = benign_samples + malicious_windows_samples + synthetic_malicious_samples
win_labels = [0] * len(benign_samples) + [1] * len(malicious_windows_samples) + [1] * len(synthetic_malicious_samples)

print("\nEncoding dataset for Windows threat detection...")
X_win_threat = model.encode(all_win_logs)
y_win_threat = np.array(win_labels)

X_wt_train, X_wt_test, y_wt_train, y_wt_test = train_test_split(
    X_win_threat, y_win_threat, test_size=0.2, random_state=42, stratify=y_win_threat
)

# Turn on `class_weight='balanced'` to better balance training
win_threat_classifier = LogisticRegression(max_iter=1000)
win_threat_classifier.fit(X_wt_train, y_wt_train)

y_wt_pred = win_threat_classifier.predict(X_wt_test)
print("\n--- Windows Threat Classifier Performance ---")
print(classification_report(y_wt_test, y_wt_pred, zero_division=0))


Encoding dataset for Windows threat detection...

--- Windows Threat Classifier Performance ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       146
           1       1.00      0.73      0.84        11

    accuracy                           0.98       157
   macro avg       0.99      0.86      0.92       157
weighted avg       0.98      0.98      0.98       157



In [12]:
import pandas as pd

# 1. Create a curated batch of test logs (mix of benign and malicious)
test_review_logs = [
    # Benign audit/authentication events
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4624, Category: Authentication, Event: An account was successfully logged on. Security ID: S-1-5-18",
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4907, Category: Configuration, Event: Auditing settings on an object were changed.",
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4634, Category: Authentication, Event: An account was logged off. Account Name: AdminUser",
    
    # Malicious/Threat events
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4625, Category: Security, Event: An account failed to log on for user Administrator from remote source IP 192.168.1.150. Reason: Bad password.",
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4688, Category: Detailed Tracking, Event: A new process has been created: C:\\Temp\\mimikatz.exe spawned by cmd.exe with command line: privilege::debug",
    "Source: Microsoft-Windows-Security-Auditing, EventID: 4697, Category: Service Installation, Event: A service was installed in the system: Service Name: PersistenceAgent | File Path: C:\\Windows\\Temp\\evil_payload.exe"
]

# Build temporary dataframe from test logs
test_df = pd.DataFrame({"Event": test_review_logs})

# Extract EventID
test_df["EventID"] = test_df["Event"].str.extract(r"EventID:\s*(\d+)")

# Extract original Category
test_df["Category"] = test_df["Event"].str.extract(r"Category:\s*([^|]+)")

# Generate expected categories using training logic
test_df["ExpectedCategory"] = test_df.apply(
    assign_windows_category,
    axis=1
)

print("Running batch analysis on test logs...\n" + "="*80)

# 2. Encode the test logs using your SentenceTransformer model
test_embeddings = model.encode(test_review_logs)

# 3. Get predictions from both classifiers
predicted_categories = win_classifier.predict(test_embeddings)
threat_predictions = win_threat_classifier.predict(test_embeddings)

# 4. Format and print the analysis report for student review
for i, log in enumerate(test_review_logs):
    category = predicted_categories[i]
    is_threat = threat_predictions[i]
    expected_category = test_df.iloc[i]["ExpectedCategory"]
    
    # Map binary threat label to human-readable text
    threat_status = "🚨 MALICIOUS THREAT" if is_threat == 1 else "✅ Benign"
    match = "✅" if category == expected_category else "❌"
    
    print(f"Log Sample #{i + 1}")
    print(f"Raw Log String : {log}")
    print(f"Expected Cat   : {expected_category}")
    print(f"Assigned Cat   : {category} {match}")
    print(f"Threat Analysis: {threat_status}")
    print("-" * 80)

Running batch analysis on test logs...
Log Sample #1
Raw Log String : Source: Microsoft-Windows-Security-Auditing, EventID: 4624, Category: Authentication, Event: An account was successfully logged on. Security ID: S-1-5-18
Expected Cat   : Authentication
Assigned Cat   : Authentication ✅
Threat Analysis: ✅ Benign
--------------------------------------------------------------------------------
Log Sample #2
Raw Log String : Source: Microsoft-Windows-Security-Auditing, EventID: 4907, Category: Configuration, Event: Auditing settings on an object were changed.
Expected Cat   : Configuration
Assigned Cat   : Configuration ✅
Threat Analysis: ✅ Benign
--------------------------------------------------------------------------------
Log Sample #3
Raw Log String : Source: Microsoft-Windows-Security-Auditing, EventID: 4634, Category: Authentication, Event: An account was logged off. Account Name: AdminUser
Expected Cat   : Authentication
Assigned Cat   : Configuration ❌
Threat Analysis: ✅ Benig

### --- 5. RUN A LIVE SUMMARY REPORT ---

In [14]:
import pandas as pd
import numpy as np

def analyze_new_log_file(file_path):
    print(f"📥 Loading logs from: {file_path}...")
    df_new = pd.read_csv(file_path)
    
    # 1. Ensure required columns exist (adjust names based on your CSV schema)
    # Expected columns: Source, EventID, Category, Event (or Event)
    event_col = 'Event' if 'Event' in df_new.columns else 'Event'
    
    # 2. Build the exact same normalized string format used during training
    print("⚙️ Normalizing log strings...")
    df_new['normalized_log'] = df_new.apply(
        lambda row: f"Source: {row.get('Source', 'Unknown')} | EventID: {row.get('EventID', 'N/A')} | Category: {row.get('Category', 'Unknown')} | Event: {str(row.get(event_col, ''))[:300]}",
        axis=1
    )
    
    log_texts = df_new['normalized_log'].tolist()
    
    # 3. Generate embeddings using your SentenceTransformer model
    print("🔮 Generating vector embeddings...")
    embeddings = model.encode(log_texts, show_progress_bar=True)
    
    # 4. Predict categories and threats using your trained models
    print("🤖 Running multi-class classification and threat scoring...")
    df_new['Predicted_Category'] = win_classifier.predict(embeddings)
    
    # Optional: Use probability thresholding if you want to control false positives
    # probs = win_threat_classifier.predict_proba(embeddings)[:, 1]
    # df_new['Threat_Score'] = probs
    # df_new['Is_Threat'] = [1 if p > 0.75 else 0 for p in probs]
    
    df_new['Is_Threat'] = win_threat_classifier.predict(embeddings)
    
    # 5. Print Summary Statistics
    print("\n" + "="*60)
    print("📊 BATCH ANALYSIS SUMMARY REPORT")
    print("="*60)
    print(f"Total Logs Analyzed : {len(df_new)}")
    print(f"Malicious Threats   : {df_new['Is_Threat'].sum()}")
    print("\nCategory Breakdown:")
    print(df_new['Predicted_Category'].value_counts())
    print("="*60)
    
    # 6. Isolate and display malicious threats
    threats_df = df_new[df_new['Is_Threat'] == 1]
    
    if not threats_df.empty:
        print(f"\n🚨 ALERT: {len(threats_df)} MALICIOUS THREAT(S) DETECTED FOR REVIEW:")
        print("-" * 60)
        for idx, row in threats_df.iterrows():
            print(f"Row Index    : {idx}")
            print(f"Event ID     : {row.get('EventID', 'N/A')}")
            print(f"Category     : {row['Predicted_Category']}")
            print(f"Log Event  : {str(row.get(event_col, ''))[:150]}...")
            print("-" * 60)
    else:
        print("\n✅ Clean Run: No malicious threats detected in this batch.")
        
    return df_new

# --- Example Usage ---
analyzed_results = analyze_new_log_file("../logs/Windows_Security/_PC3.csv")

📥 Loading logs from: ../logs/Windows_Security/_PC3.csv...
⚙️ Normalizing log strings...
🔮 Generating vector embeddings...


Batches:   0%|          | 0/965 [00:00<?, ?it/s]

🤖 Running multi-class classification and threat scoring...

📊 BATCH ANALYSIS SUMMARY REPORT
Total Logs Analyzed : 30856
Malicious Threats   : 0

Category Breakdown:
Predicted_Category
Configuration     28083
Authentication     2129
Plain Text          644
Name: count, dtype: int64

✅ Clean Run: No malicious threats detected in this batch.


# Model Comparison

Compare testing dataset against different models.

Identify differences in accuracy and runtime.

### Load Models

In [13]:
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoModelForCausalLM

# flan
flan_name = "google/flan-t5-base"

flan_tokenizer = AutoTokenizer.from_pretrained(
    flan_name,
    cache_dir=MODELS_DIR
)

flan_model = AutoModelForSeq2SeqLM.from_pretrained(
    flan_name,
    cache_dir=MODELS_DIR
).to(DEVICE)

# Qwen
qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_name,
    cache_dir=MODELS_DIR
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    cache_dir=MODELS_DIR
).to(DEVICE)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

### Classifiers

In [14]:
def classify_flan(log_text):
    # Modify prompt to adjust results
    prompt = f"""
Classify this Windows log into one category only:

Authentication
Configuration
Security
Service
Plain Text

Example:
Log: User successfully logged on.
Category: Authentication

Example:
Log: Auditing settings changed.
Category: Configuration

Example:
Log: Service installed in system.
Category: Service

Now classify this log.

Log:
{log_text}

Category:
"""
    inputs = flan_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)
    # Adjust max_new_tokens to minimize output (originally 20)
    with torch.no_grad():
        outputs = flan_model.generate(
            **inputs,
            max_new_tokens=20
        )
    return flan_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    ).strip()

def classify_qwen(log_text):
    messages = [
        {
            "role": "system",
            "content":
            "Return only one category: Authentication, Configuration, Security, Service, Plain Text"
        },
        {
            "role": "user",
            "content": log_text
        }
    ]
    prompt = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = qwen_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=20
        )

    answer_tokens = outputs[0][inputs.input_ids.shape[1]:]
    return qwen_tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip()


### Benchmark

In [15]:
def benchmark_logistic(df):
    correct = 0
    invalid = 0

    VALID_CATEGORIES = [
        "Authentication",
        "Configuration",
        "Interface",
        "Plain Text",
        "Routing",
        "Security",
        "Service",
    ]

    # Start timer for embedding and inference
    start = time.perf_counter()
    embeddings = model.encode(
        df["FullLog"].tolist()
    )
    predictions = win_classifier.predict(
        embeddings
    )
    
    # End timer
    runtime = (
        time.perf_counter() - start
    )
    
    for predicted, expected in zip(
        predictions,
        df["ExpectedCategory"]):
        if predicted == expected:
            correct += 1
        # else:              # Uncomment for debugging
        #     print("Expected category : ", row["ExpectedCategory"])
        #     print("Predicted category: ",prediction)
        #     # print("Event", row["Event"])
        #     print("*"*80)
        
        # Count invalid responses
        if predicted not in VALID_CATEGORIES:
            invalid += 1

    return {
        "Model": "Logistic Regression",
        "Events": len(df),
        "Correct": correct,
        "Invalid": invalid,
        "Accuracy": correct / len(df),
        "Runtime": runtime,
        "Events/sec": len(df) / runtime
    }

In [16]:
def benchmark_flan(df):
    VALID_CATEGORIES = [
        "Authentication",
        "Configuration",
        "Interface",
        "Plain Text",
        "Routing",
        "Security",
        "Service",
    ]
    
    start = time.perf_counter()
    correct = 0
    invalid = 0

    for _, row in df.iterrows():
        prediction = classify_flan(
            row["FullLog"]
        )[0]

        if prediction.strip().lower() == row["ExpectedCategory"].strip().lower():
            correct += 1
        # else:              # Uncomment for debugging
        #     print("Expected category : ", row["ExpectedCategory"])
        #     print("Predicted category: ",prediction)
        #     # print("Event", row["Event"])
        #     print("*"*80)

        # Count invalid responses
        if prediction not in VALID_CATEGORIES:
            invalid += 1

    runtime = (
        time.perf_counter() - start
    )

    return {
        "Model": "FLAN-T5",
        "Events": len(df),
        "Correct": correct,
        "Invalid": invalid,
        "Accuracy": correct / len(df),
        "Runtime": runtime,
        "Events/sec": len(df) / runtime
    }

In [17]:
def benchmark_qwen(df):
    VALID_CATEGORIES = [
        "Authentication",
        "Configuration",
        "Interface",
        "Plain Text",
        "Routing",
        "Security",
        "Service",
    ]
    
    start = time.perf_counter()
    correct = 0
    invalid = 0

    for _, row in df.iterrows():
        prediction = classify_qwen(
            row["FullLog"]
        )
        if prediction.strip().lower() == row["ExpectedCategory"].strip().lower():
            correct += 1
        # else:              # Uncomment for debugging
        #     print("Expected category : ", row["ExpectedCategory"])
        #     print("Predicted category: ",prediction)
        #     # print("Event", row["Event"])
        #     print("*"*80)

        # Count invalid responses
        if prediction not in VALID_CATEGORIES:
            invalid += 1
            
    runtime = (
        time.perf_counter() - start
    )

    return {
        "Model": "Qwen 1.5B",
        "Events": len(df),
        "Correct": correct,
        "Invalid": invalid,
        "Accuracy": correct / len(df),
        "Runtime": runtime,
        "Events/sec": len(df) / runtime
    }

### Run Benchmark

This may take several minutes. Larger files and non-GUI servers will take longer.

In [ ]:
import pandas as pd
import time
from pathlib import Path

benchmark_file = "../logs/Windows_Security/benchmark_small.csv"
benchmark_df = pd.read_csv(benchmark_file)

print(
    f"Loaded {len(benchmark_df)} benchmark events"
)

# Get expected categories
benchmark_df["ExpectedCategory"] = benchmark_df.apply(
    assign_windows_category,
    axis=1
)
# Combine into a single log entity
benchmark_df["FullLog"] = (
    benchmark_df["DateTime"].astype(str) + "," +
    benchmark_df["Source"].astype(str) + "," +
    benchmark_df["EventID"].astype(str) + "," +
    benchmark_df["Category"].astype(str) + "," +
    benchmark_df["Event"].astype(str)
)

# # Benchmark custom model
# results = benchmark_logistic(benchmark_df)
# display(results)

# # Benchmark flan
# results = benchmark_flan(benchmark_df)
# display(results)

# Benchmark Qwen
results = benchmark_qwen(benchmark_df)
display(results)



Loaded 29 benchmark events
